# J-Quants API V2 スタートガイド

本ノートブックでは、J-Quants API V2 の Python クライアントライブラリ `jquants-api-client` の使用方法について解説します。

## 本ノートブックの内容

- J-Quants API V2 の認証方法（API キーの発行・設定）
- `ClientV2` クラスを使用したデータ取得の実行例
- 株価データの可視化


# 1. 環境構築


In [ ]:
# jquants-api-client のインストール
%pip install jquants-api-client


# 2. J-Quants API V2 の認証方法

J-Quants API V2 では、API キー（x-api-key）による認証方式を採用しています。

## 2.1 API キーの発行手順

1. [J-Quants API サービス登録ページ](https://jpx-jquants.com/auth/signup) よりサービスへ登録します。
2. [ログインページ](https://jpx-jquants.com/auth/signin) から登録したメールアドレス及びパスワードでログインします。
3. [API キー管理ページ](https://jpx-jquants.com/dashboard/api-keys) にアクセスし、API キーを発行します。
4. 発行された API キーをコピーして安全な場所に保管してください。

※ API キーは第三者に漏洩しないよう厳重に管理してください。

## 2.2 API キーの設定方法

API キーは以下のいずれかの方法で設定できます。

### 方法 1: 引数で直接指定

```python
cli = jquantsapi.ClientV2(api_key="your-api-key")
```

### 方法 2: 環境変数で指定

```bash
export JQUANTS_API_KEY="your-api-key"
```

```python
cli = jquantsapi.ClientV2()  # 環境変数から自動取得
```

### 方法 3: 設定ファイル（jquants-api.toml）で指定

`~/.jquants-api/jquants-api.toml` または `./jquants-api.toml` に以下の内容を記載します。

```toml
[jquants-api-client]
api_key = "your-api-key"
```


In [ ]:
# Google Colab を使用する場合は、Google Drive をマウントして設定ファイルを読み込むことも可能です
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings

import jquantsapi

# pandas データフレームの表示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 80)

warnings.simplefilter('ignore')

# ClientV2 の初期化
# 設定ファイルまたは環境変数に API キーが設定されている場合は引数不要
cli = jquantsapi.ClientV2()

# 引数で直接指定する場合は以下のように記述します
# cli = jquantsapi.ClientV2(api_key="your_api_key")


# 3. 上場銘柄一覧 API

東京証券取引所に上場している銘柄の情報を取得します。


In [ ]:
# 上場銘柄一覧を取得
df_master = cli.get_eq_master()

# データフレームの情報を確認
df_master.info()

print("\n")

# データフレームを表示
df_master


In [ ]:
# セクター情報付きの銘柄一覧を取得
df_list = cli.get_list()
df_list


## 3.1 市場区分別・業種別の銘柄数


In [ ]:
# 市場区分別の銘柄数
df_market = df_list.groupby("MktNmEn").size().to_frame(
    "Count").reset_index().sort_values(by="Count", ascending=False)
px.bar(df_market, x="MktNmEn", y="Count", title="市場区分別の銘柄数")


In [ ]:
# ETF/REIT 等を除外した 17 業種別の銘柄数
df = df_list[df_list["S17"] != "99"]
df_sector = df.groupby("S17NmEn").size().to_frame(
    "Count").reset_index().sort_values(by="Count", ascending=False)
px.bar(df_sector, x="S17NmEn", y="Count", title="17業種別の銘柄数")


# 4. 株価日足 API

株価データを取得します。調整済み株価（株式分割・併合を考慮）と調整前の株価の両方を取得できます。


In [ ]:
# 特定銘柄の株価を取得（例: JPX 8697）
df_price = cli.get_eq_bars_daily(code="86970")

# データフレームの情報を確認
df_price.info()

print("\n")

# データフレームを表示
df_price


In [ ]:
# 日付範囲を指定して全銘柄の株価を取得
d_from = datetime.now() - timedelta(days=30)
d_to = datetime.now() - timedelta(days=1)

df_prices = cli.get_eq_bars_daily_range(
    start_dt=d_from.strftime(format="%Y%m%d"),
    end_dt=d_to.strftime(format="%Y%m%d")
)
df_prices


## 4.1 株価チャートの描画

取得した株価データをローソク足チャートとして可視化します。


In [ ]:
# JPX（8697）の株価を取得
df_jpx = cli.get_eq_bars_daily(code="86970")

# グラフに第2軸を設定
fig = make_subplots(specs=[[{"secondary_y": True}]])

# ローソク足を描画
fig.add_trace(
    go.Candlestick(
        x=df_jpx.Date,
        open=df_jpx.AdjO,
        high=df_jpx.AdjH,
        low=df_jpx.AdjL,
        close=df_jpx.AdjC,
        name="OHLC"
    )
)

# 25日移動平均線
fig.add_trace(
    go.Scatter(
        x=df_jpx.Date,
        y=df_jpx.AdjC.rolling(25).mean(),
        name="25日移動平均線",
        line=dict(color="#FF6B6B")
    )
)

# 75日移動平均線
fig.add_trace(
    go.Scatter(
        x=df_jpx.Date,
        y=df_jpx.AdjC.rolling(75).mean(),
        name="75日移動平均線",
        line=dict(color="#4ECDC4")
    )
)

# 出来高を第2軸に設定
fig.add_trace(
    go.Bar(
        x=df_jpx.Date,
        y=df_jpx.AdjVo,
        name="出来高",
        marker_color="rgba(128, 128, 128, 0.3)"
    ),
    secondary_y=True
)

fig.update_layout(
    title_text="JPX（8697）株価チャート",
    xaxis_title="日付",
    yaxis_title="株価（円）",
    yaxis2_title="出来高"
)
fig.show()


# 5. 決算情報 API

上場会社の四半期及び通期の決算短信に係る情報を取得します。


In [ ]:
# 特定銘柄の決算情報を取得
df_fins = cli.get_fin_summary(code="86970")

# データフレームの情報を確認
df_fins.info()

print("\n")

# データフレームを表示
df_fins


## 5.1 キャッシュを使用した範囲取得

`get_fin_summary_range()` メソッドでは、`cache_dir` 引数を指定することで、取得したデータをローカルにキャッシュできます。
2回目以降の実行ではキャッシュが利用されるため、API 呼び出し回数を削減できます。


In [ ]:
import os
from datetime import datetime, timedelta

# キャッシュディレクトリを指定（例: ホームディレクトリ配下）
cache_dir = os.path.expanduser("~/.jquants/cache")

# 日付範囲を指定して決算情報を取得（キャッシュ有効）
d_from = datetime.now() - timedelta(days=30)
d_to = datetime.now() - timedelta(days=1)

df_fins_range = cli.get_fin_summary_range(
    start_dt=d_from.strftime("%Y%m%d"),
    end_dt=d_to.strftime("%Y%m%d"),
    cache_dir=cache_dir  # キャッシュを有効化
)

print(f"取得件数: {len(df_fins_range)}")
print(f"キャッシュ保存先: {cache_dir}")
df_fins_range.head()


# 6. 指数 API（Light プラン以上）

TOPIX 等の指数データを取得します。


In [ ]:
# TOPIX 指数を取得
df_topix = cli.get_idx_bars_daily_topix()

df_topix.info()

print("\n")

df_topix


In [ ]:
# TOPIX 指数の推移をプロット
fig = px.line(
    df_topix,
    x="Date",
    y="C",
    title="TOPIX 指数の推移"
)
fig.update_layout(
    xaxis_title="日付",
    yaxis_title="TOPIX"
)
fig.show()
